# Energy Minimization of Lennard-Jones charged particles — Steepest Descent

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook performs a simple **energy minimization (EM)** of a 2D system of
Lennard-Jones particles that may also carry a charge (Coulomb interaction).
Starting from a random arrangement, EM walks the system *downhill* on its potential-energy
surface towards a nearby local minimum. The minimizer here is a **steepest-descent**
algorithm with an adaptive step size.

## Theory in brief

### Lennard-Jones

$$E_{LJ}(r) \;=\; 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^{6}\right]$$

The $(\sigma/r)^{12}$ term is the steep short-range **repulsion** (overlapping atoms), the
$-(\sigma/r)^{6}$ term the weaker long-range **attraction**. The two cancel exactly at
$r=\sigma$ (`Sigma`), and the energy reaches its minimum $-\varepsilon$ (`Epsilon`) a little
further out, at $R_{min}=2^{1/6}\sigma\approx1.12\,\sigma$. So `Sigma` sets the particle
*size* and `Epsilon` the well *depth*. The code evaluates this same expression from the
**squared** distance, so the inner loop needs no square root.

To save work, pairs beyond a cutoff (`CutOff`) are ignored — this makes the potential
slightly discontinuous at the cutoff, which is harmless here.

*(For the potential on its own — the two components plotted separately, in real force-field
units — see [`LJ-ELEC_Potentials`](LJ-ELEC_Potentials.ipynb).)*

### Coulomb

$$E_{Coul} = \frac{q_a q_b}{\epsilon_r\, r}$$

Like charges repel ($E>0$), unlike charges attract ($E<0$); the dielectric constant
`Dielec` ($\epsilon_r$) screens (weakens) the interaction.

### Minimization
The **force** on each atom is minus the gradient of the total energy, i.e. the local
downhill direction. **Steepest descent** moves every particle straight along the
(normalised) total force $\mathbf{F}$, by a step `dr`. The step is scaled **up** by `alpha`
when the energy decreases and **down** by `beta` when it increases; iteration stops when the
energy change, the step size, or the force norm drops below its threshold.

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`), so it runs
on a bare Python install. **matplotlib** is the one third-party dependency — it draws the
static figures and the trajectory animation (embedded inline as interactive HTML via
`jshtml`). The cell below first **installs matplotlib if it is missing** (handy on Google
Colab), then imports everything; `%matplotlib inline` renders figures inside the notebook.

In [ ]:
# --- Install required packages (works locally, on Colab, and on JupyterLite) ---
%pip install -q matplotlib

from math import sqrt

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, seed

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. Helper functions

Small utilities used throughout:

* `dist` / `dist2` — Euclidean distance and its square (the squared form avoids a needless
  `sqrt` when we only need to compare distances).
* `SignR(a, b)` — returns `a` with the sign of `b`. It implements the **minimum-image
  convention**: the combination `tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)`
  wraps a coordinate difference into the range $[-\tfrac{box}{2}, +\tfrac{box}{2}]$, so each
  atom interacts with the *nearest periodic copy* of its neighbours (periodic boundary
  conditions).
* `charge_color` — purely cosmetic: white for positive charges, dark for negative, used when
  drawing the particles.

In [ ]:
### distance ###
def dist(A, B):
    return sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

### squared distance ###
def dist2(A, B):
    return (A[0]-B[0])**2 + (A[1]-B[1])**2

### change sign ###
def SignR(a, b):
    if b > 0:
        return a
    else:
        return -a

### colour particles based on charge ###
def charge_color(charge, qat):
    if charge == qat:
        return "#FFFFFF"   # positive
    else:
        return "#333333"   # negative

## 3. Energy functions

Total energy is the sum over all particle pairs of the Lennard-Jones and Coulomb
contributions, using the **nearest image** convention (periodic boundary conditions).

Distances are handled as **squared** distances (`distsquare`) throughout the inner loops:
this avoids computing a `sqrt` for every pair, and lets the cutoff test (`distsquare <
cutoffsquare`) skip distant pairs cheaply. A `sqrt` is taken only where a term actually
needs $r$ itself (the Coulomb $1/r$).

In [ ]:
# LJ energy from the squared distance
def LJ2(distsquare, epsilon, sigma_exp6):
    # E_LJ = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ], from the squared distance
    u = (1/distsquare)**3 * sigma_exp6        # u = (sigma/r)^6,  u*u = (sigma/r)^12
    return 4*epsilon * u * (u - 1)            # = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ]

# classical Coulomb from the squared distance
def Coulomb2(r, dielec, qa, qb):
    return qa*qb / (dielec*sqrt(r))

# Calculate energy Evdw + Ecoulomb (uses squared distance), with periodic boundary conditions
def Calc_Ene2(coord, epsilon, sigma, dielec, cutoffsquare, boxdim, elec=1):
    Ene = 0.0
    ELJ = 0.0
    ECoul = 0.0
    sigma_exp6 = sigma**6
    # doubly nested loop over all particle pairs
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                vdw = LJ2(distsquare, epsilon, sigma_exp6)
                Ene += vdw
                ELJ += vdw
                if elec:
                    CC = Coulomb2(distsquare, dielec, qa, qb)
                    Ene += CC
                    ECoul += CC
    return Ene, ELJ, ECoul

## 4. Force functions

The force on an atom is minus the gradient of the total energy — the local *downhill*
direction. For the Lennard-Jones term the component along $x$ is obtained by the chain rule,

$$F_x = -\frac{\partial E}{\partial x} = -\frac{\partial E}{\partial u}\,
        \frac{\partial u}{\partial r}\,\frac{\partial r}{\partial x},
\qquad u=\left(\frac{\sigma}{r}\right)^{6},\quad E_{LJ}=4\varepsilon\,(u^{2}-u),$$

which maps directly onto the code:

* `dedu` $= \partial E/\partial u = 4\varepsilon\,(2u-1)$
* `dudr` $= \partial u/\partial r = -6\,\sigma^{6}/r^{7}$
* `drdx` $= x_i/r$, where `xi` $= x_j - x_i$ — using the $j-i$ difference already carries the
  sign that turns $-\partial E/\partial x_i$ into the force on atom $i$.

The Coulomb force follows the same pattern, with `dedr` $= -q_a q_b/(\epsilon_r\, r^{2})$.

In [ ]:
# LJ force component (uses squared distance)
def ForceLJ2(distsquare, epsilon, sigma_exp6, xi):
    # E_LJ = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ] = 4 eps (u^2 - u) with u = (sigma/r)^6
    rij = sqrt(distsquare)
    u    = (1/distsquare)**3 * sigma_exp6
    dedu = 4*epsilon*(2*u - 1)                # dE/du
    dudr = sigma_exp6*(-6.0/rij**7.0)         # du/dr
    drdx = xi/rij
    return dedu*dudr*drdx

# Coulomb force component (uses squared distance)
def ForceCoulomb2(distsquare, dielec, qa, qb, xi):
    rij = sqrt(distsquare)
    dedr = -1.0*(qa*qb/dielec)*(1/distsquare)
    drdx = xi/rij
    return dedr*drdx

# Total force on each atom from Evdw + Ecoulomb (uses squared distance)
def Calc_Force2(coord, epsilon, sigma, dielec, cutoffsquare, boxdim):
    Force = []
    sigma_exp6 = sigma**6
    for i in range(len(coord)):
        tmpforce = [0.0, 0.0]
        for j in range(len(coord)):
            if i == j:
                continue
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                fflist = []
                for k in range(2):
                    tmp = coord[j][k] - coord[i][k]
                    ff = ForceLJ2(distsquare, epsilon, sigma_exp6, tmp)
                    ff += ForceCoulomb2(distsquare, dielec, qa, qb, tmp)
                    fflist.append(ff)
                for k in range(2):
                    tmpforce[k] = tmpforce[k] + fflist[k]
        Force.append(tmpforce)
    return Force

## 5. The steepest-descent minimizer

This is the core of the minimiser. Given the current coordinates, the step size `drstep`
and the forces, it:

1. computes the norm of the total force vector, then
2. moves every particle a distance `drstep` along the **normalised** force direction.

It returns the new coordinates and the force norm.

In [ ]:
def Steepest_descent(atom_coord, drstep, force):
    """Steepest-descent update.

    Parameters
    ----------
    atom_coord : list of [x, y, q] for each atom
    drstep     : displacement for the minimizer
    force      : list of [fx, fy] for each atom

    Returns
    -------
    new_coord, normf
    """
    newlist = []

    # 1) norm of the total force vector
    normf = 0.0
    for i in range(len(atom_coord)):
        normf = normf + force[i][0]**2.0 + force[i][1]**2.0
    normf = sqrt(normf)

    # 2) move the particles along the normalised force
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        if normf > 0:
            sx = force[i][0]/normf
            sy = force[i][1]/normf
            r0x = r0x + drstep*sx
            r0y = r0y + drstep*sy
        newlist.append([r0x, r0y, q])
    return newlist, normf

## 6. Parameters

These are the same parameters exposed by the sliders/entry boxes of the original GUI.
Change any of them and re-run **this cell together with the Initialisation and Run cells just below** to explore their effect (or use *Kernel → Restart & Run All*). Values are in the
toy model's arbitrary units (lengths in box/canvas units, energies loosely in kcal/mol).

### System and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nAtoms` | number of particles | 2–40 |
| `Radius` | particle radius (drawn size, and default absolute charge) | 10–40 — must leave room to place all atoms, or initialisation fails |
| `Sigma` | LJ distance parameter $\sigma$: the separation where $E_{LJ}=0$ (the minimum sits at $2^{1/6}\sigma$) | `2.24 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth (the minimum of $E_{LJ}$ is $-$`Epsilon`) | 0.25–25 |
| `Dielec` | dielectric constant (charge screening) | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | defaults to `Radius` |
| `frac_neg` | fraction of negative charges | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |

### Minimizer

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `drinit` | initial step size `dr` | 0.1–5 |
| `drmin` / `drmax` | min / max allowed `dr` | 1e-5 / 5 |
| `alpha` / `beta` | scale `dr` up / down after each step | 1.05 / 0.90 |
| `deltaE` | energy-change stop threshold (tighter -> settles into a deeper minimum) | 1e-5 |
| `normFmin` | force-norm stop threshold | 1e-4 |
| `max_iter` | hard cap on the number of steps | 2000 |

In [ ]:
nAtoms  = 20              # number of atoms
Radius  = 25.0            # atom radius (must be in a sensible range vs nAtoms so all atoms fit)
Sigma   = 2.24 * Radius   # LJ distance parameter sigma (E_LJ = 0 here; minimum at 2^(1/6)*Sigma)
BoxDim  = [500, 500]      # box dimensions
Epsilon = 6.25            # LJ well depth (the minimum of E_LJ is -Epsilon)
Dielec  = 1.0             # dielectric constant
qat     = Radius          # atom absolute charge
frac_neg = 0.5            # fraction of negative charges
OverlapFr = 0.0           # fraction of overlap allowed when placing atoms
CutOff  = 250             # non-bonded cutoff
CutOffSquare = CutOff**2

# --- steepest-descent controls ---
drinit  = 1.00        # initial dr for EM
drmin   = 0.00001     # minimum dr value to keep stepping
drmax   = 5.00        # maximum dr
alpha   = 1.05        # scale factor for dr when Enew < Eold
beta    = 0.90        # scale factor for dr when Enew > Eold
deltaE  = 0.00001     # energy-difference threshold to stop EM (tight -> avoids stopping early on a plateau)
normFmin = 0.0001     # minimum force norm to keep stepping

Seed    = 100         # random number seed
max_iter = 2000       # safety cap on the number of EM steps

## 7. Initialisation

Generate random, non-overlapping starting positions and assign charges
(a fraction `frac_neg` negative, the rest positive).

In [ ]:
import sys

### generate random, non-overlapping coordinates ###
def InitConf(n, dim, radius, qat, frac_neg):
    seed(Seed)
    print("Initializing box, please wait...")
    tmp_coord = []
    i = 0
    ntrial = 0
    nneg = int(float(n) * frac_neg)
    npos = n - nneg

    # first atom
    x = random()*(dim[0]-2*radius) + radius
    y = random()*(dim[1]-2*radius) + radius
    charge = -qat
    if npos == n:
        charge = qat
    i += 1
    if n == 2:
        tmp_coord.append([175, 300, charge])
    else:
        tmp_coord.append([x, y, charge])

    # remaining negative charges
    while i < nneg:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = -qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed")
            print("==> reduce radius or number of atoms")
            sys.exit()

    # remaining positive charges
    while i < n:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 10**10:
            print("initialisation failed")
            print("==> reduce radius or number of atoms")
            sys.exit()
    return tmp_coord


Atom_Coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
Color = [charge_color(a[2], qat) for a in Atom_Coord]
print(f"Placed {len(Atom_Coord)} atoms.")

## 8. Run the minimization

This loop replaces the GUI's `Go` callback / Tkinter event loop. At each step it computes
the forces, takes a steepest-descent step, adaptively rescales `dr`, applies periodic
boundary conditions, and records the trajectory and energies. It stops when a convergence
criterion is met or after `max_iter` steps.

In [ ]:
def run_minimization(Atom_Coord):
    """Headless steepest-descent minimization. Returns trajectory + energy history."""
    coord = [list(a) for a in Atom_Coord]   # work on a copy
    drstep = drinit

    Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
    Ene_prev = Ene

    traj    = [ [list(a) for a in coord] ]   # snapshot per step
    E_hist  = [Ene]
    Elj_hist = [EneLJ]
    Ecoul_hist = [EneCoul]

    print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f" % (0, Ene, EneLJ, EneCoul))

    Iterations = 0
    for step in range(1, max_iter+1):
        Force = Calc_Force2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
        coord, normF = Steepest_descent(coord, drstep, Force)
        Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
        Ene_diff = Ene - Ene_prev

        # adaptive step size
        if Ene_diff < 0.0:
            drstep = min(drmax, drstep*alpha)
        else:
            drstep = drstep*beta
        Ene_prev = Ene

        # periodic boundary conditions
        for pp in range(len(coord)):
            for k in range(2):
                if coord[pp][k] < 0:
                    coord[pp][k] += BoxDim[k]
                if coord[pp][k] > BoxDim[k]:
                    coord[pp][k] -= BoxDim[k]

        Iterations = step
        normF = normF/len(coord)

        traj.append([list(a) for a in coord])
        E_hist.append(Ene)
        Elj_hist.append(EneLJ)
        Ecoul_hist.append(EneCoul)

        if step % 50 == 0:
            print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f deltaE: %10.6f <normF>: %8.6f dr: %8.6f"
                  % (step, Ene, EneLJ, EneCoul, Ene_diff, normF, drstep))

        # convergence
        if abs(Ene_diff) < deltaE or drstep < drmin or normF < normFmin:
            print("STOPPING... deltaE<%g, or drstep<%g, or normF<%g" % (deltaE, drmin, normFmin))
            print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f deltaE: %10.6f <normF>: %8.6f dr: %8.6f"
                  % (step, Ene, EneLJ, EneCoul, Ene_diff, normF, drstep))
            break

    return traj, E_hist, Elj_hist, Ecoul_hist


traj, E_hist, Elj_hist, Ecoul_hist = run_minimization(Atom_Coord)
print(f"\nDone in {len(traj)-1} steps. Final Epot = {E_hist[-1]:.2f}")

## 9. Energy convergence

How the total, Lennard-Jones and Coulomb energies evolve during the minimization.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
steps = range(len(E_hist))
ax.plot(steps, E_hist,     label="Epot (total)", lw=2)
ax.plot(steps, Elj_hist,   label="E$_{LJ}$",  lw=1.5)
ax.plot(steps, Ecoul_hist, label="E$_{Coul}$", lw=1.5)
ax.set_xlabel("EM step")
ax.set_ylabel("Energy (kcal/mol)")
ax.set_title("Steepest-descent energy minimization")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 10. Visualise the system

Start and end configurations side by side. White = positive charge, dark = negative.

In [ ]:
def draw_config(ax, coord, title):
    ax.set_xlim(0, BoxDim[0])
    ax.set_ylim(0, BoxDim[1])
    ax.set_aspect('equal')
    ax.set_facecolor("#ccddff")
    ax.set_title(title)
    ax.invert_yaxis()   # match the original canvas (y downwards)
    for a in coord:
        col = charge_color(a[2], qat)
        ax.add_patch(Circle((a[0], a[1]), Radius, facecolor=col, edgecolor="black", lw=0.8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
draw_config(ax1, traj[0],  f"Initial  (Epot = {E_hist[0]:.1f})")
draw_config(ax2, traj[-1], f"Minimized (Epot = {E_hist[-1]:.1f})")
plt.tight_layout()
plt.show()

## 11. Animation of the minimization

Replays the whole trajectory. This reproduces the live view of the original GUI.
(To keep it light, only every few frames are shown — adjust `stride`.)

In [ ]:
stride = max(1, len(traj)//120)   # cap at ~120 frames
frames = list(range(0, len(traj), stride))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#ccddff")
ax.invert_yaxis()

circles = [Circle((a[0], a[1]), Radius,
                  facecolor=charge_color(a[2], qat), edgecolor="black", lw=0.8)
           for a in traj[0]]
for c in circles:
    ax.add_patch(c)
title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    for c, a in zip(circles, traj[f]):
        c.center = (a[0], a[1])
    title.set_text(f"step {f}   Epot = {E_hist[f]:.1f}")
    return circles + [title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 12. Comparison of the three minimizers

This notebook implements **the steepest-descent minimizer**. The three companion notebooks
(`LJ-ELEC_EM-steepest_Py3`, `LJ-ELEC_EM-conjugate_Py3`, `simplex`) minimize the **same system**
— 20 particles with identical parameters and the same random seed (`Seed = 100`). Running each
**with its default parameters** gives:

| Method | Final energy | Steps |
|---|---|---|
| Steepest descent | ≈ −409 | ~1100 |
| Conjugate gradient | ≈ −383 | ~970 |
| Simplex (Nelder–Mead) | ≈ −187 | ~620 |

Take-aways:

* All three reach a **local** minimum — none is guaranteed to find the global minimum, and the
  result depends on the starting configuration (the random seed) and the path taken.
* **Steepest descent** lands in the deepest basin here; **conjugate gradient** takes fewer steps
  but, following a different path, settles in a slightly shallower minimum.
* The **simplex** is *derivative-free* (energy only, no forces) — simple and robust, but it scales
  poorly to this 40-dimensional search space (2 × 20 coordinates), so it converges to a much
  shallower minimum. A good illustration of why gradient-based methods dominate for smooth,
  high-dimensional problems.

*(Energies and step counts are for `Seed = 100` with each notebook's default parameters; other
seeds give different absolute numbers.)*

---
## **13. Exercises**

Everything above ran **one** minimization, from **one** starting configuration, with **one** set of
parameters. These three exercises take that apart: **tune** the step-size controller, **reverse**
the minimizer on purpose, and ask how much of the answer was decided by where it started.

> **How they work.** Each exercise starts from a stub containing a `TODO` placeholder. Replace it
> with your own code and re-run the cell. The cells are **self-checking**: they stay quiet until
> your code works, then fill themselves in with the numbers and plots you need. A worked solution
> is folded away under each exercise — try it yourself first.

Run the cell below once before starting. It wraps the run loop of section 8 in a function,
`run_em(...)`, whose parameters are **arguments instead of globals** — so an exercise can launch
dozens of minimizations without editing section 6. It is the same algorithm (same forces, same
adaptive `dr`, same convergence test), only quieter, plus three things the exercises need: a
`stepper` argument (Exercise 2 will supply an *ascent* stepper), an `uphill` switch for the
step-size test, and a guard that abandons a run whose energy runs away. A few small utilities
come with it: `sweep` (Exercise 1), and `pair_distance` / `neighbour_pairs` / `closest_pair`,
which measure the structure using the same nearest-image convention as the energy function.

In [ ]:
from math import isfinite

def run_em(drinit=drinit, drmax=drmax, alpha=alpha, beta=beta,
           epsilon=Epsilon, dielec=Dielec,
           stepper=Steepest_descent, uphill=False,
           deltaE=deltaE, drmin=drmin, normFmin=normFmin, max_iter=max_iter,
           coord0=None, keep_traj=False):
    """The run loop of section 8, with parameters as arguments instead of globals.

    Same algorithm: forces -> step along the normalised force -> rescale dr
    (*alpha if the energy went down, *beta if it went up, capped at drmax) ->
    periodic boundaries -> convergence test.  Differences: it prints nothing,
    the stepping rule can be replaced (`stepper`), the direction of the dr test
    can be flipped (`uphill=True` grows dr when the energy goes *up*), and the
    run is abandoned if the energy runs away.

    Returns a dict with  E (final energy), steps, E_hist, coord (final),
    traj (only if keep_traj=True) and reason ('converged' / 'max_iter' / 'blew up').
    """
    coord = [list(a) for a in (Atom_Coord if coord0 is None else coord0)]
    drstep = drinit
    sgn = -1.0 if uphill else 1.0          # which sign of dE counts as progress

    Ene, _, _ = Calc_Ene2(coord, epsilon, Sigma, dielec, CutOffSquare, BoxDim)
    Ene_prev = Ene
    E_hist = [Ene]
    traj = [[list(a) for a in coord]] if keep_traj else None
    reason, steps = 'max_iter', 0

    for step in range(1, max_iter+1):
        try:
            Force = Calc_Force2(coord, epsilon, Sigma, dielec, CutOffSquare, BoxDim)
            coord, normF = stepper(coord, drstep, Force)
            Ene, _, _ = Calc_Ene2(coord, epsilon, Sigma, dielec, CutOffSquare, BoxDim)
        except (ZeroDivisionError, OverflowError):
            reason = 'blew up'
            break
        steps = step
        E_hist.append(Ene)
        if keep_traj:
            traj.append([list(a) for a in coord])
        if not isfinite(Ene) or abs(Ene) > 1e12:      # runaway guard
            reason = 'blew up'
            break

        Ene_diff = Ene - Ene_prev
        if sgn*Ene_diff < 0.0:
            drstep = min(drmax, drstep*alpha)
        else:
            drstep = drstep*beta
        Ene_prev = Ene

        for pp in range(len(coord)):                  # periodic boundaries
            for k in range(2):
                if coord[pp][k] < 0:
                    coord[pp][k] += BoxDim[k]
                if coord[pp][k] > BoxDim[k]:
                    coord[pp][k] -= BoxDim[k]

        if abs(Ene_diff) < deltaE or drstep < drmin or normF/len(coord) < normFmin:
            reason = 'converged'
            break

    return {'E': E_hist[-1], 'steps': steps, 'E_hist': E_hist,
            'coord': coord, 'traj': traj, 'reason': reason}


def sweep(trials, **kwargs):
    """Run one minimization per (drmax, alpha, beta) triple and tabulate the outcome."""
    results = []
    for drmax_, alpha_, beta_ in trials:
        r = run_em(drmax=drmax_, alpha=alpha_, beta=beta_, **kwargs)
        r.update(drmax=drmax_, alpha=alpha_, beta=beta_)
        results.append(r)
    # a run is on the "best trade-off" front if no other run is both deeper and faster
    for r in results:
        r['front'] = not any(o['E'] < r['E'] - 1e-6 and o['steps'] < r['steps'] for o in results)
    print(f"{'drmax':>6} {'alpha':>6} {'beta':>6} {'final E':>10} {'steps':>7}  {'stopped by':<10}")
    for r in sorted(results, key=lambda r: r['steps']):
        flag = "  <-- best trade-off" if r['front'] else ""
        print(f"{r['drmax']:6g} {r['alpha']:6.2f} {r['beta']:6.2f} "
              f"{r['E']:10.2f} {r['steps']:7d}  {r['reason']:<10}{flag}")
    return results


def pair_distance(coord, i, j):
    """Nearest-image distance between atoms i and j (same convention as Calc_Ene2)."""
    d2 = 0.0
    for k in range(2):
        tmp = coord[j][k] - coord[i][k]
        halfbox = BoxDim[k]/2
        tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
        d2 += tmp**2
    return sqrt(d2)


def neighbour_pairs(coord, rcut):
    """All pairs closer than rcut, as a list of (i, j, r)."""
    out = []
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            r = pair_distance(coord, i, j)
            if r < rcut:
                out.append((i, j, r))
    return out


def closest_pair(coord):
    """Distance of the closest pair in the box."""
    return min(r for _, _, r in neighbour_pairs(coord, max(BoxDim)))


print(f"toolbox ready -- a reference run: ", end="")
_ref = run_em()
print(f"E = {_ref['E']:.2f} in {_ref['steps']} steps ({_ref['reason']})")

---
### **Exercise 1 — tune the step-size controller**

The minimizer of section 5 only ever decides *where* to go; how *far* to go is decided by three
numbers in section 6, used by the run loop of section 8:

* `alpha` (> 1) — `dr` is multiplied by this after a step that **lowered** the energy: things are
  going well, be bolder;
* `beta` (< 1) — `dr` is multiplied by this after a step that **raised** the energy: we overshot,
  back off;
* `drmax` — the ceiling `dr` may never exceed.

The defaults (`drmax = 5`, `alpha = 1.05`, `beta = 0.90`) are just *a* choice. Your job is to find
a better one: **the lowest energy in the fewest steps.** One step costs one force evaluation plus
one energy evaluation, so the step count *is* the computational cost.

**(a)** Predict, before running anything, what the minimizer does with

* `alpha = 1.00, beta = 1.00` (no adaptation at all — a fixed step of `drinit`);
* `beta = 0.95` (it barely backs off after an overshoot);
* `drmax = 100` (essentially no ceiling).

**(b)** Fill in `trials` in the cell below with 6–10 `(drmax, alpha, beta)` combinations, including
the extremes you just predicted, and run it. Every run starts from the same configuration with the
same `drinit`, so the comparison is fair.

**(c)** Read the table and the figure. Which combination reaches the **lowest** energy? Which one
converges in the **fewest steps**? Is the fastest also the deepest? Which runs never converged at
all — and what do their energy curves look like?

**(d)** Does `drmax` actually matter here? Before blaming the ceiling, look at the `dr` column
printed by section 8 during a normal run: how large does `dr` really get?

**(e)** `alpha` and `beta` are not the only knobs on cost. Re-run the sweep with a looser
convergence threshold — `sweep(trials, deltaE=1e-3)` — and look at what your best combination does
now. You buy a large cut in steps; what do you pay for it? (Section 6 has a hint in its `deltaE`
row.)

In [ ]:
# ---------- Exercise 1: which (drmax, alpha, beta) combinations do you want to test? ----------
TODO = None

trials = TODO           # <-- your code here: a list of (drmax, alpha, beta) triples
#
# For example (uncomment and extend -- add the extreme cases you predicted in (a)):
#
# trials = [(5, 1.05, 0.90),     # the notebook default
#           (5, 1.00, 1.00),     # no adaptation at all
#           ...
#          ]

# ---------- self-check ----------
if trials is None:
    results1 = None
    print("Not yet: fill in `trials` above with a few (drmax, alpha, beta) triples and re-run.")
else:
    print(f"{len(trials)} runs, all starting from the same configuration "
          f"(E = {E_hist[0]:.2f}) with drinit = {drinit}\n")
    results1 = sweep(trials)
    best_E = min(results1, key=lambda r: r['E'])
    best_n = min((r for r in results1 if r['reason'] == 'converged'),
                 key=lambda r: r['steps'], default=None)
    print(f"\ndeepest    : E = {best_E['E']:.2f} in {best_E['steps']} steps "
          f"(drmax={best_E['drmax']:g}, alpha={best_E['alpha']}, beta={best_E['beta']})")
    if best_n:
        print(f"fastest    : E = {best_n['E']:.2f} in {best_n['steps']} steps "
              f"(drmax={best_n['drmax']:g}, alpha={best_n['alpha']}, beta={best_n['beta']})")
    print("\n(one step = one force + one energy evaluation, so `steps` is the cost)")

In [ ]:
# Convergence curves and the cost/depth trade-off for the runs above
if results1 is None:
    print("Fill in `trials` in the cell above to get the figure.")
else:
    fig, (axE, axP) = plt.subplots(1, 2, figsize=(12.5, 4.8))

    for k, r in enumerate(results1, start=1):
        lab = f"#{k}  drmax={r['drmax']:g}, a={r['alpha']}, b={r['beta']}"
        axE.plot(range(len(r['E_hist'])), r['E_hist'], lw=1.5, label=lab)
    axE.set_xlabel("EM step")
    axE.set_ylabel("Epot (kcal/mol)")
    axE.set_title("the same descent, driven by different step-size controls")
    axE.grid(alpha=0.3)
    axE.legend(fontsize=7.5, loc="upper right")

    for k, r in enumerate(results1, start=1):
        style = dict(marker="o", ms=9, color="#1b8a5a") if r['front'] else dict(marker="o", ms=7, color="#6a3d9a")
        if r['reason'] != 'converged':
            style.update(marker="X", ms=9, color="#b03030")
        axP.plot(r['steps'], r['E'], **style)
        left = r['steps'] > 0.75*max(o['steps'] for o in results1)   # keep labels inside
        axP.annotate(f"#{k}", xy=(r['steps'], r['E']),
                     xytext=(-7 if left else 5, 5), textcoords="offset points",
                     ha="right" if left else "left", fontsize=9)
    axP.set_xlabel("steps to stop  (cost)")
    axP.set_ylabel("final Epot  (quality)")
    axP.set_title("cheap and deep is the bottom-left corner\n"
                  "(green = best trade-off, X = never converged; #k = row k of `trials`)",
                  fontsize=10)
    axP.margins(x=0.1, y=0.12)
    axP.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

<details>
<summary><b>Exercise 1 — worked solution and expected answers</b></summary>

**(a)** With `alpha = beta = 1` the step size never changes: the run becomes a plain fixed-step
descent that cannot fine-tune near the minimum, so it keeps rattling around it and never satisfies
`deltaE` — it runs out of iterations. With `beta = 0.95` the controller hardly punishes an
overshoot, so the step stays too large: the energy oscillates instead of settling. With
`drmax = 100` almost nothing changes, because `dr` in this system rarely grows past ~5 anyway
(see (d)).

**(b)** A representative set, and what it gives (`Seed = 100`):

```python
trials = [(5,   1.05, 0.90),   # the notebook default
          (5,   1.00, 1.00),   # no adaptation at all
          (5,   1.01, 0.50),   # timid growth, harsh cut
          (5,   1.05, 0.95),   # barely backs off after an uphill step
          (5,   1.20, 0.90),   # fast growth, gentle cut
          (10,  1.20, 0.50),   # fast growth, harsh cut
          (100, 1.05, 0.90),   # very high ceiling
          (1,   1.05, 0.90)]   # very low ceiling
```

| # | `drmax` | `alpha` | `beta` | final E | steps | stopped by |
|---|---|---|---|---|---|---|
| 6 | 10 | 1.20 | 0.50 | **−409.45** | **909** | converged |
| 7 | 100 | 1.05 | 0.90 | −409.45 | 1069 | converged |
| 1 | 5 | 1.05 | 0.90 | −409.45 | 1109 | converged |
| 8 | 1 | 1.05 | 0.90 | −409.45 | 1394 | converged |
| 2 | 5 | 1.00 | 1.00 | −409.34 | 2000 | max_iter |
| 3 | 5 | 1.01 | 0.50 | −259.60 | 2000 | max_iter |
| 4 | 5 | 1.05 | 0.95 | −409.09 | 2000 | max_iter |
| 5 | 5 | 1.20 | 0.90 | −401.57 | 2000 | max_iter |

**(c)** Four of the eight land in the **same** minimum, −409.45, so "deepest" is a four-way tie and
the question becomes which of them is cheapest: the basin is set by the starting configuration and
the downhill direction, not by the step size — the step size decides **how quickly** you get there,
and **whether you can stop**. The winner is `(10, 1.20, 0.50)`: grow boldly while things improve,
then cut the step hard on the first overshoot — 909 steps, ~18 % cheaper than the default for
exactly the same minimum. The failures are instructive:

* `beta = 0.95` and `alpha = beta = 1` never converge — the step is never brought down far enough
  for `deltaE` to be satisfied, so they oscillate *around* the minimum (at −409.09 and −409.34 they
  are essentially there, they just cannot tell) until `max_iter` stops them. `(1.20, 0.90)` is the
  same disease one notch worse: a bold `alpha` with a soft `beta` overshoots faster than it
  recovers, and it is still wandering at −401.57 after 2000 steps;
* `(1.01, 0.50)` is the opposite failure: `dr` is halved at every overshoot and crawls back at
  1 % per step, so the run grinds to a near-standstill far up the surface and is still at −259.6
  when `max_iter` cuts it off — its energy curve (green in the figure) is visibly still sloping
  downwards. That is a **stalled** run: it did not find a minimum, it ran out of step size. Had
  `deltaE` been a little looser it would have reported `converged` — which is the trap of
  Exercise 2(d).

The pattern: **grow fast, cut hard.** A bold `alpha` costs nothing as long as `beta` can undo the
damage in one step.

**(d)** Barely. The printout of section 8 shows `dr` mostly wandering between ~0.05 and ~4.5, so
the ceiling of 5 is rarely reached and lifting it to 100 changes almost nothing: 1069 steps against
1109, the same minimum. (The two runs differ *at all* only because `dr` does occasionally touch 5.)
`drmax` is a safety belt against a single catastrophic step, not a performance knob — but lower it
to 1 and it really does bind: 1394 steps for the same answer.

**(e)** With `deltaE = 1e-3` the winning combination stops after **488** steps instead of 909 — a
46 % saving — but at **−396.06** instead of −409.45. It has halted on a **plateau**, not in the
minimum: close to a minimum the energy flattens out long before the forces vanish, so a loose
energy threshold is satisfied while the structure is still relaxing (the default combination does
the same: 659 steps, −399.18). Cheap convergence is easy; the question is always *converged to
what*. This is why the notebook uses a tight `deltaE = 1e-5` and also checks the force norm
(`normFmin`) — the gradient is the honest convergence test, the energy difference only a cheap
proxy.

</details>

---
### **Exercise 2 — run it backwards: steepest *ascent***

A minimizer walks downhill because we *told* it to: `Steepest_descent` adds `drstep` times the
normalised force, and the force is *minus* the gradient. Flip that sign and the very same machinery
climbs. Let's see what it finds.

**(a)** Write `Steepest_ascent` in the cell below: same signature and same return values as
`Steepest_descent` (section 5), but moving each atom **against** the force. The self-check verifies
that one ascent step is the exact mirror image of one descent step.

Note that **two** things have to be reversed to climb properly, not one: the move *and* the
step-size test — `dr` should now grow when the energy goes **up**. That is what `run_em`'s
`uphill=True` does, and the self-check runs both variants so you can see the difference.

**(b)** Predict first: if the algorithm maximises this energy, where do the particles end up? Which
way do two *unlike* charges move? Two *like* charges? Look at the two terms in section 3 and ask
which one wins at short range.

**(c)** Run the cell. How many steps does the ascent survive, and what stops it? What is the
closest pair distance at the end, compared with `Sigma` (the diameter of a particle)?

**(d)** The self-check also runs the **half-reversed** version — ascent stepper, descent step-size
test. It reports `converged`. Is it? What is the energy when it "converges", and what has actually
happened to `dr`? (Moral: never trust a convergence flag on its own.)

**(e)** So: *is* there a maximum to find? Write down what happens to the Lennard-Jones and Coulomb
terms as $r \to 0$ and decide which one dominates. What does that say about the energy surface —
and about asking an optimiser to find something that does not exist?

**(f)** Finally, a one-line alternative: you can maximise $E$ by minimising $-E$, i.e. by flipping
the sign of the **force** instead of writing a new stepper. Convince yourself that this is the same
computation. (Which is why "minimizers" and "maximisers" are never separate programs in practice —
and why a molecular-dynamics or docking code that gets one sign wrong does not crash, it just
produces nonsense.)

In [ ]:
# ---------- Exercise 2: write the ascent stepper ----------
TODO = None

def Steepest_ascent(atom_coord, drstep, force):
    """Steepest *ascent*: move each atom drstep *against* the normalised force.

    Same signature and same return values as Steepest_descent (section 5):
    (new_coord, normf), where new_coord is a list of [x, y, q].
    """
    return TODO                                    # <-- your code here


# ---------- self-check ----------
_F0 = Calc_Force2(Atom_Coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)

def solved_ascent():
    """True once Steepest_ascent returns coordinates instead of the TODO placeholder."""
    try:
        new, _ = Steepest_ascent([list(a) for a in Atom_Coord], 1.0, _F0)
        return new is not None and len(new) == len(Atom_Coord)
    except Exception:
        return False

if not solved_ascent():
    up = None
    print("Not yet: replace the TODO in Steepest_ascent above and re-run this cell.")
else:
    # (a) one step of ascent must be exactly minus one step of descent
    down, _ = Steepest_descent([list(a) for a in Atom_Coord], 1.0, _F0)
    upc,  _ = Steepest_ascent([list(a) for a in Atom_Coord], 1.0, _F0)
    mirror = max(abs((u[k]-a[k]) + (d[k]-a[k]))
                 for a, d, u in zip(Atom_Coord, down, upc) for k in (0, 1))
    print(f"(a) first step is the mirror image of the descent step: "
          f"largest mismatch {mirror:.2e}  -> {'OK' if mirror < 1e-9 else 'check the sign'}\n")

    # (c) the real thing: uphill stepper AND uphill step-size test
    up = run_em(stepper=Steepest_ascent, uphill=True)
    print(f"(c) steepest ascent: stopped after {up['steps']} steps ({up['reason']})")
    print(f"    energy  {up['E_hist'][0]:.2f}  ->  {up['E']:.3g}")
    print("    step:  " + "  ".join(f"{i}:{e:.3g}" for i, e in
                                    list(enumerate(up['E_hist']))[:9]) + "  ...")
    growth = [up['E_hist'][i+1]/up['E_hist'][i]
              for i in range(5, min(12, len(up['E_hist'])-1))]
    print(f"    early on the energy grows by a factor "
          f"{min(growth):.1f}-{max(growth):.1f} per step, then faster and faster")
    print(f"    closest pair: {closest_pair(Atom_Coord):.1f} at the start  ->  "
          f"{closest_pair(up['coord']):.2f} at the end   (Sigma = {Sigma:.1f})")
    nsq = len(neighbour_pairs(up['coord'], 0.5*Sigma))
    print(f"    pairs closer than Sigma/2 = {0.5*Sigma:.1f}: "
          f"{len(neighbour_pairs(Atom_Coord, 0.5*Sigma))} at the start, {nsq} at the end")

    # (d) change the stepper but forget the dr test
    half = run_em(stepper=Steepest_ascent, uphill=False)
    print(f"\n(d) ascent stepper + the *descent* dr test: {half['steps']} steps, "
          f"reported '{half['reason']}', final E = {half['E']:.1f}")
    print("    -> it claims convergence. Look at the energy before you believe it.")

In [ ]:
# Downhill versus uphill, from the same starting configuration
if up is None:
    print("Complete Steepest_ascent in the cell above to get the figure.")
else:
    fig = plt.figure(figsize=(12.5, 4.8))
    axE = fig.add_subplot(1, 3, 1)
    axA = fig.add_subplot(1, 3, 2)
    axC = fig.add_subplot(1, 3, 3)

    axE.plot(range(len(E_hist)), E_hist, color="#1b8a5a", lw=1.8)
    axE.axhline(0, color="#999999", lw=0.8)
    axE.set_xlabel("EM step"); axE.set_ylabel("Epot (kcal/mol)")
    axE.set_title(f"descent: converges\n{len(E_hist)-1} steps, E = {E_hist[-1]:.1f}", fontsize=10)
    axE.grid(alpha=0.3)

    first = next(i for i, e in enumerate(up['E_hist']) if e > 0)
    axA.semilogy(range(first, len(up['E_hist'])), up['E_hist'][first:],
                 color="#b03030", lw=1.8, marker="o", ms=3)
    axA.set_xlabel("EM step"); axA.set_ylabel("Epot (kcal/mol, log scale)")
    axA.set_title(f"ascent: diverges\n{up['steps']} steps, E = {up['E']:.1g}", fontsize=10)
    axA.grid(alpha=0.3, which="both")
    axA.annotate(f"steps 0-{first-1} are still\nnegative (not shown\non a log axis)",
                 xy=(first, up['E_hist'][first]), xytext=(first+0.6, 1e8),
                 fontsize=8, color="#555555",
                 arrowprops=dict(arrowstyle="->", color="#555555", lw=0.8))

    draw_config(axC, up['coord'], "after ascent: two atoms have merged\n"
                                  f"(red ring: 2 atoms, {closest_pair(up['coord']):.1f} apart "
                                  f"-- Sigma = {Sigma:.0f})")
    i, j, _ = min(neighbour_pairs(up['coord'], max(BoxDim)), key=lambda p: p[2])
    axC.add_patch(Circle((up['coord'][i][0], up['coord'][i][1]), 2.4*Radius,
                         fill=False, edgecolor="#b03030", lw=2.0))

    plt.tight_layout()
    plt.show()

<details>
<summary><b>Exercise 2 — worked solution and expected answers</b></summary>

**(a)**

```python
def Steepest_ascent(atom_coord, drstep, force):
    newlist = []
    normf = 0.0
    for i in range(len(atom_coord)):
        normf = normf + force[i][0]**2.0 + force[i][1]**2.0
    normf = sqrt(normf)
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        if normf > 0:
            r0x = r0x - drstep*force[i][0]/normf      # minus: against the force
            r0y = r0y - drstep*force[i][1]/normf
        newlist.append([r0x, r0y, q])
    return newlist, normf
```

Only the two signs change — everything else, including the force norm, is identical.

**(b, c)** The ascent **diverges**: starting from E = −66.11 it is abandoned after **21 steps**
with E ≈ 7.3 × 10¹², when the runaway guard in `run_em` fires. Early on the energy grows by a
factor of ~1.7 per step, then faster and faster — an exponential blow-up, not a convergence.
The closest pair goes from 52.3 at the start to **6.2** — about a ninth of `Sigma = 56`, the two
discs drawn on top of each other in the right-hand panel: the "maximum" the algorithm is chasing is
a pair of atoms in the same place.
Which way does an individual pair go? Uphill means undoing whatever made the energy low: an
attracting unlike-charge pair is stretched **apart**, while any pair close enough to feel the
repulsive wall is driven **into** it. And since that wall is by far the steepest feature of the
surface, it soon dominates the normalised direction and drags the whole run into the collapse
above.

**(d)** With the descent step-size test the energy goes **up** at every step, so `dr` is multiplied
by `beta` every time and collapses geometrically. After 110 steps it is so small that
`|ΔE| < deltaE` and the run happily reports `converged` — at **E ≈ +1015**, a thousand kcal/mol
*above* where it started. Nothing converged: the step size died. A convergence flag says "I stopped
moving", which is only the same as "I found a stationary point" if the step size was not the thing
that stopped you. This is the same trap as the stalled `(1.01, 0.50)` run in Exercise 1.

**(e)** As $r \to 0$ the Coulomb term goes to $\pm\infty$ like $1/r$ while the Lennard-Jones
repulsion goes to $+\infty$ like $r^{-12}$ — so the repulsion always wins, even for an
oppositely-charged pair, and the total energy is **unbounded above**. There is no maximum. The
surface has plenty of minima (any relaxed packing) and saddle points between them, but climbing it
has no destination: the optimiser does exactly what it was asked to do, for ever. Worth
remembering the next time an optimisation "fails to converge" — sometimes the answer you asked for
does not exist. (The [soft core](LJ-ELEC_MD-SoftCore.ipynb) notebook shows the same $r^{-12}$ wall
blowing up an MD run, and how capping it rescues the simulation.)

**(f)** Minimising $-E$ means stepping along $-\nabla(-E) = +\nabla E = -F$ — which is precisely
"move against the force". Same arithmetic, so the same trajectory; only the bookkeeping differs.
In practice you flip one sign in the force (or in the objective) and reuse the whole minimizer.

</details>

---
### **Exercise 3 — how local is "local"?**

Every number this notebook has produced comes from **one** starting configuration, the one
`Seed = 100` happens to generate. A minimizer can only walk downhill from where it is put, so the
minimum it reaches is a property of that starting point as much as of the algorithm. This is the
single most important caveat about energy minimisation — and it is a few lines of code away.

**(a)** Predict first, and write your guess down: run the same minimizer on ten different random
starting configurations. How far apart will the ten final energies be — a fraction of a kcal/mol,
a few, tens? Would you expect the *best starting* configuration to give the *deepest* minimum?

**(b)** Write `minimize_with_seed(s)` in the cell below. The catch: `InitConf()` takes no seed
argument — it reads the **global** `Seed` (look at section 7), so you have to set that first.
Then build a configuration and pass it to `run_em(coord0=...)`.

**(c)** Run it for `SEEDS = range(100, 110)` and read the table: the spread between the deepest and
the shallowest minimum, the mean and the standard deviation. Where does `Seed = 100` — the
configuration every section above used, and the −409 quoted in section 12 — rank among the ten?

**(d)** Look at the second panel of the figure, and at the correlation the cell prints. Does
starting from a *lower* energy lead to a *lower* minimum? Why (not)?

**(e)** Compare the deepest and the shallowest structure in the last figure. Are they the same
arrangement seen twice, or genuinely different packings? Count the separate clusters in each. Which
one would you call *the* minimized structure of this system — and why can the minimizer not repair
the other one?

**(f)** Now re-read the comparison table in section 12: one seed per method. What does this exercise
say about that ranking — and about any paper that reports "the minimized energy" of anything? What
would you have to do to make the comparison honest? (Increase `SEEDS` to `range(100, 130)` if you
want to see the spread with thirty starting points instead of ten.)

In [ ]:
# ---------- Exercise 3: minimize from a different starting configuration ----------
TODO = None

def minimize_with_seed(s):
    """Build a fresh random configuration with random seed `s`, minimize it, and
    return run_em's result dictionary.

    Three lines. Remember that InitConf() does not take a seed: it reads the *global* `Seed`,
    so you have to point that at `s` first (`global Seed`). Then build the configuration with
    InitConf(nAtoms, BoxDim, Radius, qat, frac_neg) and hand it to run_em(coord0=...).
    """
    return TODO                                    # <-- your code here


# ---------- self-check ----------
import io, contextlib, statistics

SEEDS = list(range(100, 110))          # ten starting configurations
_Seed_notebook = Seed                  # so we can put it back afterwards

def quiet(f, *args):
    """Call f while swallowing InitConf's progress messages."""
    with contextlib.redirect_stdout(io.StringIO()):
        return f(*args)

def solved_seed():
    try:
        return quiet(minimize_with_seed, Seed) is not None
    except Exception:
        return False

if not solved_seed():
    seed_runs = None
    print("Not yet: replace the TODO in minimize_with_seed above and re-run this cell.")
else:
    print(f"minimizing {len(SEEDS)} different starting configurations "
          f"(same parameters, same minimizer)...\n")
    seed_runs = []
    for s in SEEDS:
        r = quiet(minimize_with_seed, s)
        r['seed'] = s
        seed_runs.append(r)
    Seed = _Seed_notebook              # leave the notebook's own seed as we found it

    print(f"{'seed':>6} {'start E':>10} {'final E':>10} {'steps':>7}  stopped by")
    for r in seed_runs:
        mark = "   <-- this notebook" if r['seed'] == _Seed_notebook else ""
        print(f"{r['seed']:6d} {r['E_hist'][0]:10.2f} {r['E']:10.2f} {r['steps']:7d}  "
              f"{r['reason']}{mark}")

    finals = [r['E'] for r in seed_runs]
    starts = [r['E_hist'][0] for r in seed_runs]
    deepest = min(seed_runs, key=lambda r: r['E'])
    shallowest = max(seed_runs, key=lambda r: r['E'])
    rank = sorted(finals).index(next(r['E'] for r in seed_runs if r['seed'] == _Seed_notebook)) + 1

    print(f"\n  mean  {statistics.mean(finals):8.1f}   sd {statistics.stdev(finals):6.1f}")
    print(f"  best  {deepest['E']:8.1f}  (seed {deepest['seed']})")
    print(f"  worst {shallowest['E']:8.1f}  (seed {shallowest['seed']})")
    print(f"  spread {shallowest['E'] - deepest['E']:7.1f} kcal/mol "
          f"-- {(shallowest['E']-deepest['E'])/abs(statistics.mean(finals))*100:.0f} % of the mean")
    print(f"\n  Seed = {_Seed_notebook} (the one this notebook uses) ranks {rank} of {len(SEEDS)}")

    # does a good starting point lead to a good minimum?
    mx, my = statistics.mean(starts), statistics.mean(finals)
    cov = sum((a-mx)*(b-my) for a, b in zip(starts, finals))
    den = sqrt(sum((a-mx)**2 for a in starts) * sum((b-my)**2 for b in finals))
    print(f"  correlation between starting and final energy: r = {cov/den:+.2f}")

In [ ]:
# Where do ten different starting configurations end up?
if seed_runs is None:
    print("Complete minimize_with_seed in the cell above to get the figure.")
else:
    finals = [r['E'] for r in seed_runs]
    starts = [r['E_hist'][0] for r in seed_runs]
    mean_E = statistics.mean(finals)

    fig, (axB, axS) = plt.subplots(1, 2, figsize=(12.5, 4.8))

    order = sorted(seed_runs, key=lambda r: r['E'])
    labels = [str(r['seed']) for r in order]
    colors = ["#1b8a5a" if r['seed'] == _Seed_notebook else "#6a3d9a" for r in order]
    axB.bar(labels, [r['E'] for r in order], color=colors)
    axB.axhspan(min(finals), max(finals), color="#6a3d9a", alpha=0.10, zorder=0)
    axB.text(0.01, max(finals), f" spread {max(finals)-min(finals):.0f} kcal/mol",
             transform=axB.get_yaxis_transform(), ha="left", va="bottom",
             color="#4b2d6a", fontsize=9)
    axB.axhline(mean_E, color="#b03030", ls="--", lw=1.2)
    axB.text(0.99, mean_E, f" mean {mean_E:.0f}", transform=axB.get_yaxis_transform(),
             ha="right", va="bottom", color="#b03030", fontsize=9)
    axB.set_xlabel("random seed  (sorted by final energy)")
    axB.set_ylabel("final Epot (kcal/mol)")
    axB.set_title(f"same system, same minimizer, {len(seed_runs)} starting points\n"
                  f"(green = Seed {_Seed_notebook}, the one this notebook uses)", fontsize=10)
    axB.set_ylim(min(finals)*1.06, 0)
    axB.grid(alpha=0.3, axis="y")

    axS.plot(starts, finals, "o", ms=9, color="#6a3d9a")
    for r in seed_runs:
        axS.annotate(str(r['seed']), xy=(r['E_hist'][0], r['E']), xytext=(5, 4),
                     textcoords="offset points", fontsize=7.5)
    axS.set_xlabel("energy of the starting configuration")
    axS.set_ylabel("final Epot (kcal/mol)")
    axS.set_title("a better starting point does not buy a better minimum", fontsize=10)
    axS.margins(x=0.13, y=0.13)
    axS.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# The best and the worst minimum, side by side (white = +, dark = -)
if seed_runs is None:
    print("Complete minimize_with_seed in the cell above to get the figure.")
else:
    deepest = min(seed_runs, key=lambda r: r['E'])
    shallowest = max(seed_runs, key=lambda r: r['E'])
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    for ax, r, tag in ((axes[0], deepest, "deepest"), (axes[1], shallowest, "shallowest")):
        draw_config(ax, r['coord'],
                    f"{tag}: seed {r['seed']}, Epot = {r['E']:.1f}\n"
                    f"({r['steps']} steps, closest pair {closest_pair(r['coord']):.1f})")
    plt.tight_layout()
    plt.show()

<details>
<summary><b>Exercise 3 — worked solution and expected answers</b></summary>

**(b)**

```python
def minimize_with_seed(s):
    global Seed                       # InitConf reads the global, not an argument
    Seed = s
    coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
    return run_em(coord0=coord)
```

**(c)** Seeds 100-109, everything else untouched:

| seed | start E | final E | steps |
|---|---|---|---|
| 100 | −66.11 | **−409.45** | 1109 |
| 101 | −103.50 | −334.92 | 1391 |
| 102 | −38.25 | −312.98 | 601 |
| 103 | −52.46 | −355.59 | 1549 |
| 104 | −108.52 | −302.65 | 696 |
| 105 | −139.43 | −362.02 | 1237 |
| 106 | −74.85 | −322.37 | 568 |
| 107 | −52.76 | −340.27 | 528 |
| 108 | −115.57 | −355.39 | 937 |
| 109 | −94.42 | −338.45 | 790 |

mean **−343.4**, standard deviation **30.1**, best −409.4, worst −302.7 — a spread of **107
kcal/mol**, about **31 %** of the mean. And the punchline: **`Seed = 100` is the best of the ten.**
The −409 this notebook has been quoting all along, including in the comparison table of section 12,
is not a typical result — it is the luckiest of ten. A typical run of this system lands near −340.
(With thirty seeds it gets worse: mean −312 ± 57, worst −193, and seed 100 is *still* the best of
the thirty.)

Note also that the step count varies just as much (528 to 1549) and is unrelated to the depth
reached: seed 103 works 1549 steps to reach −355.6, seed 107 gets −340.3 in a third of that.

**(d)** No: the correlation between the starting energy and the final energy is **r ≈ +0.04** —
nothing. Seed 105 starts lowest (−139.4) and finishes mid-pack; seed 100 starts almost highest
(−66.1) and finishes deepest. Steepest descent is a purely **local** procedure: it follows the
gradient into whichever basin it happens to be standing in, and how favourable the initial
configuration looks says nothing about how deep that basin goes. "Downhill from here" is not
"downhill overall".

**(e)** Genuinely different, and the figure shows *why* the worst one is bad: seed 100 condenses
into a **single compact cluster**, while seed 104 freezes as **three separate fragments** scattered
across the box. Every particle in those fragments is already at the bottom of its local well — the
forces have vanished — but the fragments are far enough apart that almost nothing pulls them
together, and merging them would mean dragging whole groups across empty space. There is no
*downhill* path from three clusters to one, so a minimizer, which only ever goes downhill, can
never repair it. That missing contact between fragments is exactly the ~107 kcal/mol.

Both are *correct* local minima of the same energy function, so there is no such thing as "*the*
minimized structure" here: there is one per starting point, and the one you report is the one you
happened to start from.

**(f)** Section 12 compares steepest descent (−409), conjugate gradient (−383) and simplex (−187)
on a **single** seed. The first two differ by 26 kcal/mol — well inside the 107 kcal/mol spread
this exercise just measured, so that difference is **not evidence** that one minimizer is better
than the other: rerun with `Seed = 104` and the ranking could flip. (The simplex's −187 is a
different matter: it is far outside the spread, so its weakness is real.) An honest comparison
minimizes the **same set** of many starting configurations with each method and compares the
distributions — mean, best-of-N, and how many function evaluations each needed.

That is also how the problem is attacked in practice: you do not trust one minimisation. You run
many starts (*multi-start* minimisation), or you let the system cross barriers before minimising
again — which is exactly what the [Monte Carlo](LJ-ELEC_MMC.ipynb) and
[molecular dynamics](LJ-ELEC_MD-Verlet.ipynb) notebooks do, and what *basin hopping* (below)
automates.

</details>

</details>

---
### **Going further**

Open-ended, no scaffolding provided:

* **Change the physics.** `Dielec` (1 = vacuum, ~80 = water) and `Epsilon` set the **balance**
  between the two interactions. Minimize with `Dielec` = 1, 4, 20, 80, and separately with
  `Epsilon` = 0.25, 1, 25. Energies from different parameter sets are not comparable, so measure
  the **structure** instead — for example the fraction of neighbouring pairs (say within
  1.3·`Sigma`) carrying *opposite* charges. Does the cluster order like a salt, or pack like an
  uncharged liquid? And since only the *ratio* $q^2/(\varepsilon_r\sigma\epsilon)$ matters, what
  should happen if you multiply `Epsilon` by 4 and divide `Dielec` by 4 at the same time?
* **Beat the other minimizers.** Section 12 compares the three methods **with their default
  parameters** and a single seed. Tune the steepest-descent parameters as in Exercise 1, then do
  the same for the [conjugate-gradient](LJ-ELEC_EM-conjugate.ipynb) and
  [simplex](LJ-ELEC_EM-simplex.ipynb) notebooks — and compare them over the ten seeds of
  Exercise 3 rather than one. Does the ranking survive?
* **A real line search.** `alpha`/`beta` is a crude one-dimensional search. Replace it: along the
  chosen direction, try a few step sizes and keep the best (or fit a parabola through three
  energies). Count *energy evaluations*, not steps — is it worth it?
* **Escape the basin.** Add a small random kick to every atom whenever the run converges, then
  minimize again and keep the result only if it is lower (a minimal *basin hopping*). Starting from
  `Seed = 104`, can you reach the −409 of `Seed = 100`?
* **All charges the same sign.** Set `frac_neg = 0` and minimize. What does the system do, and what
  role do the periodic boundaries and the `CutOff` play in the answer?
* **Squeeze the box.** Keep `nAtoms = 20` but halve `BoxDim`. At what density does the minimizer
  start to struggle, and why? (`Radius` must still leave room for the initialisation to place all
  the atoms.)